In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

# Prepare Target Variables

In this section, target variable dataset (stock price data) is prepared. 

The resulting panel data dataset will serve as a base for the analysis

## Import Scraped Data

In [2]:
# Load all stock data files from scraping/output_data/stock_data
stock_data_dir = Path('../scraping/output_data/stock_data')

# List all CSV files
csv_files = sorted(stock_data_dir.glob('yahoofinance_*.csv'))
print(f"Found {len(csv_files)} stock data files")

# Load and combine all files
dfs = []
for file in csv_files:
    # Extract ticker from filename (after 'yahoofinance_' and before '.csv')
    ticker = file.stem.replace('yahoofinance_', '')
    
    # Read the CSV file
    df = pd.read_csv(file)
    
    # Add ticker column as the leftmost column
    df.insert(0, 'ticker', ticker)
    
    dfs.append(df)

# Combine all dataframes
stock_data = pd.concat(dfs, ignore_index=True)

print(f"\nCombined dataset shape: {stock_data.shape}")
print(f"Columns: {stock_data.columns.tolist()}")
print(f"Unique tickers: {stock_data['ticker'].nunique()}")
print(f"Number of trading days covered: {stock_data['Date'].nunique()}")
print(f"\nFirst few rows:")
stock_data.head(10)

Found 162 stock data files

Combined dataset shape: (390127, 8)
Columns: ['ticker', 'Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']
Unique tickers: 162
Number of trading days covered: 3514

First few rows:


,ticker,Date,Open,High,Low,Close,Adj Close,Volume
0,AALLON,"Dec 30, 2025",10.55,10.65,10.55,10.65,10.65,78
1,AALLON,"Dec 29, 2025",10.45,10.65,10.40,10.50,10.50,"2,712"
2,AALLON,"Dec 23, 2025",10.60,10.60,10.40,10.40,10.40,"2,868"
3,AALLON,"Dec 22, 2025",10.50,10.60,10.40,10.45,10.45,"2,368"
4,AALLON,"Dec 19, 2025",10.70,10.70,10.45,10.60,10.60,"5,029"
5,AALLON,"Dec 18, 2025",10.65,10.70,10.65,10.70,10.70,396
6,AALLON,"Dec 17, 2025",10.90,10.90,10.55,10.70,10.70,"8,749"
7,AALLON,"Dec 16, 2025",10.90,10.90,10.90,10.90,10.90,1
8,AALLON,"Dec 15, 2025",10.60,10.90,10.35,10.90,10.90,"3,471"
9,AALLON,"Dec 12, 2025",10.40,10.85,10.40,10.85,10.85,"1,653"


## Add Micro Control Variables

In [3]:
# Convert Date to datetime and sort data
stock_data['Date'] = pd.to_datetime(stock_data['Date'])
stock_data = stock_data.sort_values(['ticker', 'Date']).reset_index(drop=True)

# Remove commas from Volume and convert to numeric
stock_data['Volume'] = stock_data['Volume'].astype(str).str.replace(',', '').astype(float)

# Process each ticker separately to maintain temporal relationships
results = []
for ticker in stock_data['ticker'].unique():
    ticker_df = stock_data[stock_data['ticker'] == ticker].copy()
    
    # 1. Calculate returns (daily)
    ticker_df['return'] = ticker_df['Close'].pct_change()
    
    # 2. Lagged returns: t-1, t-2, t-5
    ticker_df['return_lag1'] = ticker_df['return'].shift(1)
    ticker_df['return_lag2'] = ticker_df['return'].shift(2)
    ticker_df['return_lag5'] = ticker_df['return'].shift(5)
    
    # 3. Log(Volume) - more robust measure of volume in analysis
    ticker_df['log_volume'] = np.log(ticker_df['Volume'].replace(0, np.nan))
    
    # 4. Garman-Klass volatili1ty (intraday volatility estimator)
    # GK = 0.5 * (log(High/Low))^2 - (2*log(2)-1) * (log(Close/Open))^2
    gk_var = (
        0.5 * np.log(ticker_df['High'] / ticker_df['Low'])**2
        - (2*np.log(2) - 1) * np.log(ticker_df['Close'] / ticker_df['Open'])**2
    )
    ticker_df['gk_vol'] = np.sqrt(np.maximum(gk_var, 0))
    
    # 5. 12-month Momentum (12-1): Return from t-252 to t-21 (252 trading days to 21 trading day ago)
    ticker_df['momentum_12_1'] = (
        ticker_df['Close'].shift(21) / ticker_df['Close'].shift(252)
    ) - 1

    # 6. Amihud illiquidity: 5-day rolling mean of |return|/euro_volume (t-1 to t-5)
    # Using lagged window to avoid contemporaneous correlation with the return target
    euro_vol = (ticker_df['Volume'] * ticker_df['Close']).replace(0, np.nan)
    daily_illiq = (np.abs(ticker_df['return']) / euro_vol) * 1e6
    ticker_df['amihud'] = daily_illiq.shift(1).rolling(window=5, min_periods=1).mean()

    
    # 7. Open-to-close return (intraday return)
    ticker_df['open_to_close_return'] = (ticker_df['Close'] / ticker_df['Open']) - 1
    
    # 8. Close-to-open return (overnight return)
    ticker_df['close_to_open_return'] = (ticker_df['Open'] / ticker_df['Close'].shift(1)) - 1
    
    results.append(ticker_df)

    # Data hygiene checks
    # OHLC must be positive, High ≥ Low
    mask = (ticker_df[['Open','High','Low','Close']] > 0).all(axis=1) & (ticker_df['High'] >= ticker_df['Low'])
    ticker_df = ticker_df[mask].copy()


# Combine all processed data
stock_data = pd.concat(results, ignore_index=True)

print(f"Dataset shape: {stock_data.shape}")
print(f"Columns: {stock_data.columns.tolist()}")
print(f"\nMissing values per column:")
print(stock_data.isnull().sum())
print(f"\nSample data for first ticker (sorted by date):")
print(stock_data[stock_data['ticker'] == stock_data['ticker'].iloc[0]].head(15))

Dataset shape: (390127, 18)
Columns: ['ticker', 'Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'return', 'return_lag1', 'return_lag2', 'return_lag5', 'log_volume', 'gk_vol', 'momentum_12_1', 'amihud', 'open_to_close_return', 'close_to_open_return']

Missing values per column:
ticker                      0
Date                        0
Open                        0
High                        0
Low                         0
Close                       0
Adj Close                   0
Volume                      0
return                    162
return_lag1               324
return_lag2               486
return_lag5               972
log_volume                  0
gk_vol                      0
momentum_12_1           39887
amihud                    324
open_to_close_return        0
close_to_open_return      162
dtype: int64

Sample data for first ticker (sorted by date):
    ticker       Date  Open  High   Low  Close  Adj Close    Volume    return  \
0   AALLON 2019-04-08  8.

In [4]:
# Descriptive statistics for control variables
control_vars = ['return', 'return_lag1', 'return_lag2', 'return_lag5', 'log_volume', 
                'gk_vol', 'momentum_12_1', 'amihud', 
                'open_to_close_return', 'close_to_open_return']

print("\nDescriptive statistics for control variables:")
print(stock_data[control_vars].describe())


Descriptive statistics for control variables:
              return    return_lag1    return_lag2    return_lag5  \
count  389965.000000  389803.000000  389641.000000  389155.000000   
mean        0.000272       0.000266       0.000264       0.000263   
std         0.026559       0.026554       0.026525       0.026533   
min        -0.770321      -0.770321      -0.761223      -0.761223   
25%        -0.010508      -0.010514      -0.010518      -0.010526   
50%         0.000000       0.000000       0.000000       0.000000   
75%         0.009967       0.009960       0.009953       0.009957   
max         1.777465       1.777465       1.777465       1.777465   

          log_volume         gk_vol  momentum_12_1         amihud  \
count  390127.000000  390127.000000  350240.000000  389803.000000   
mean        9.939872       0.017237       0.065410      13.411138   
std         2.659326       0.016363       0.480629     413.732670   
min         0.000000       0.000000      -0.978177     

In [5]:
stock_data.head(10)

,ticker,Date,Open,High,Low,Close,Adj Close,Volume,return,return_lag1,return_lag2,return_lag5,log_volume,gk_vol,momentum_12_1,amihud,open_to_close_return,close_to_open_return
0,AALLON,2019-04-08,8.21,8.96,8.21,8.87,7.84,104367.0,NaN,NaN,NaN,NaN,11.555669,0.038876,NaN,NaN,0.080390,NaN
1,AALLON,2019-04-09,8.90,8.90,8.46,8.60,7.60,23533.0,-0.030440,NaN,NaN,NaN,10.066159,0.028830,NaN,NaN,-0.033708,0.003382
2,AALLON,2019-04-10,8.60,8.60,8.47,8.47,7.48,11544.0,-0.015116,-0.030440,NaN,NaN,9.353921,0.005136,NaN,0.150406,-0.015116,0.000000
3,AALLON,2019-04-11,8.45,8.50,8.35,8.50,7.51,6043.0,0.003542,-0.015116,-0.030440,NaN,8.706656,0.012044,NaN,0.152502,0.005917,-0.002361
4,AALLON,2019-04-12,8.50,8.55,8.35,8.50,7.51,3199.0,0.000000,0.003542,-0.015116,NaN,8.070594,0.016737,NaN,0.124653,0.000000,0.000000
5,AALLON,2019-04-15,8.60,8.60,8.50,8.50,7.51,1598.0,0.000000,0.000000,0.003542,NaN,7.376508,0.003944,NaN,0.093490,-0.011628,0.011765
6,AALLON,2019-04-16,8.60,8.60,8.45,8.45,7.47,2226.0,-0.005882,0.000000,0.000000,-0.030440,7.707962,0.005933,NaN,0.074792,-0.017442,0.011765
7,AALLON,2019-04-17,8.45,8.55,8.45,8.55,7.56,2649.0,0.011834,-0.005882,0.000000,-0.015116,7.881937,0.003967,NaN,0.107257,0.011834,0.000000
8,AALLON,2019-04-18,8.60,8.60,8.40,8.45,7.47,4214.0,-0.011696,0.011834,-0.005882,0.003542,8.346168,0.012540,NaN,0.180839,-0.017442,0.005848
9,AALLON,2019-04-23,8.45,8.50,8.35,8.50,7.51,3995.0,0.005917,-0.011696,0.011834,0.000000,8.292799,0.012044,NaN,0.232740,0.005917,0.000000


## Add daily macro (market) variables

In [6]:
# Load macro control variables (currency pairs and commodities)
control_var_dir = Path('../scraping/output_data/control_variables')

# Define currency pairs and commodities to load
macro_vars = {
    'EURUSD': 'yahoofinance_EURUSD=X.csv',
    'EURSEK': 'yahoofinance_EURSEK=X.csv',
    'EURCNY': 'yahoofinance_EURCNY=X.csv',
    'GC': 'yahoofinance_GC=F.csv',  # Gold
    'CL': 'yahoofinance_CL=F.csv',  # Crude Oil
    'VIX': 'yahoofinance_^VIX.csv',  # Volatility Index
    'GSPC': 'yahoofinance_^GSPC.csv',  # S&P 500
    'TNX': 'yahoofinance_^TNX.csv',  # 10-Year Treasury Yield
}

# Load and process each macro variable
macro_data_list = []
for var_name, filename in macro_vars.items():
    file_path = control_var_dir / filename
    if file_path.exists():
        df_macro = pd.read_csv(file_path)
        
        # Clean and convert Date
        df_macro['Date'] = pd.to_datetime(df_macro['Date'])
        
        # Remove commas from Close and convert to numeric
        df_macro['Close'] = df_macro['Close'].astype(str).str.replace(',', '').astype(float)
        
        # Sort by date
        df_macro = df_macro.sort_values('Date')
        
        # Calculate previous day return
        df_macro[f'{var_name}_return'] = df_macro['Close'].pct_change()
        
        # Keep only Date and return columns
        df_macro = df_macro[['Date', f'{var_name}_return']]
        
        macro_data_list.append(df_macro)
        print(f"✓ Loaded {var_name}: {len(df_macro)} rows")
    else:
        print(f"⚠ File not found: {filename}")

# Merge all macro variables on Date
macro_data = macro_data_list[0]
for df in macro_data_list[1:]:
    macro_data = macro_data.merge(df, on='Date', how='outer')

macro_data = macro_data.sort_values('Date').reset_index(drop=True)

print(f"\nCombined macro data shape: {macro_data.shape}")
print(f"Date range: {macro_data['Date'].min()} to {macro_data['Date'].max()}")
print(f"\nMacro variables columns:")
print(macro_data.columns.tolist())
print(f"\nSample macro data:")
macro_data.head(10)

✓ Loaded EURUSD: 3644 rows
✓ Loaded EURSEK: 3646 rows
✓ Loaded EURCNY: 3646 rows
✓ Loaded GC: 3518 rows
✓ Loaded CL: 3513 rows
✓ Loaded VIX: 3519 rows
✓ Loaded GSPC: 3519 rows
✓ Loaded TNX: 3518 rows

Combined macro data shape: (3650, 9)
Date range: 2012-01-02 00:00:00 to 2025-12-31 00:00:00

Macro variables columns:
['Date', 'EURUSD_return', 'EURSEK_return', 'EURCNY_return', 'GC_return', 'CL_return', 'VIX_return', 'GSPC_return', 'TNX_return']

Sample macro data:


,Date,EURUSD_return,EURSEK_return,EURCNY_return,GC_return,CL_return,VIX_return,GSPC_return,TNX_return
0,2012-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2012-01-03,-0.001466,0.001459,-0.003704,NaN,NaN,NaN,NaN,NaN
2,2012-01-04,0.008886,-0.002802,0.010452,0.007626,0.002525,-0.032651,0.000188,0.017857
3,2012-01-05,-0.009574,-0.005720,-0.009308,0.004653,-0.013660,-0.033303,0.002944,-0.001003
4,2012-01-06,-0.010440,0.000475,-0.010330,-0.002038,-0.002456,-0.039572,-0.002537,-0.016056
5,2012-01-09,-0.009065,-0.003231,-0.003144,-0.005321,-0.002462,0.021328,0.002262,-0.000510
6,2012-01-10,0.007571,-0.002052,0.001970,0.014619,0.009180,-0.018035,0.008886,0.006122
7,2012-01-11,-0.001879,0.000136,0.003272,0.005028,-0.013400,0.017400,0.000310,-0.034483
8,2012-01-12,-0.002196,0.000715,-0.008011,0.004941,-0.017547,-0.027553,0.002337,0.015231
9,2012-01-13,0.007623,0.006866,0.005888,-0.010259,-0.004036,0.021495,-0.004948,-0.041386


In [7]:
# Load additional control variables
# 1. Euribor 3-month (difference, not return)
euribor_path = Path('./data/euribor_3m.pkl')
if euribor_path.exists():
    df_euribor = pd.read_pickle(euribor_path)
    df_euribor['Date'] = pd.to_datetime(df_euribor['Date'])
    df_euribor = df_euribor.sort_values('Date')
    
    # Convert euribor_3m to numeric
    df_euribor['euribor_3m'] = pd.to_numeric(df_euribor['euribor_3m'], errors='coerce')
    
    # Calculate difference (not return)
    df_euribor['euribor_3m_diff'] = df_euribor['euribor_3m'].diff()
    
    # Keep only Date and difference
    df_euribor = df_euribor[['Date', 'euribor_3m_diff']]
    
    # Merge with macro_data
    macro_data = macro_data.merge(df_euribor, on='Date', how='left')
    print(f"✓ Loaded Euribor 3m: {len(df_euribor)} rows")
else:
    print(f"⚠ File not found: {euribor_path}")

# 2. Load European/Nordic indices
index_vars = {
    'OMXHPI': 'nasdaq_^OMXHPI.csv',
    'OMXN40': 'nasdaq_^OMXN40.csv',
    'OMXH25': 'nasdaq_^OMXH25.csv',
    'STOXX50E': 'yahoofinance_^STOXX50E.csv',
}

for var_name, filename in index_vars.items():
    file_path = control_var_dir / filename
    if file_path.exists():
        df_index = pd.read_csv(file_path)
        
        # Clean and convert Date
        df_index['Date'] = pd.to_datetime(df_index['Date'])
        
        # Remove commas from Close and convert to numeric
        df_index['Close'] = df_index['Close'].astype(str).str.replace(',', '').astype(float)
        
        # Sort by date
        df_index = df_index.sort_values('Date')
        
        # Calculate return (without lag)
        df_index[f'{var_name}_return'] = df_index['Close'].pct_change()
        
        # Keep only Date and return columns
        df_index = df_index[['Date', f'{var_name}_return']]
        
        # Merge with macro_data
        macro_data = macro_data.merge(df_index, on='Date', how='outer')
        print(f"✓ Loaded {var_name}: {len(df_index)} rows")
    else:
        print(f"⚠ File not found: {filename}")

macro_data = macro_data.sort_values('Date').reset_index(drop=True)

print(f"\nUpdated macro data shape: {macro_data.shape}")
print(f"All columns: {macro_data.columns.tolist()}")
print(f"\nSample data with new variables:")
macro_data.head(10)

✓ Loaded Euribor 3m: 3577 rows
✓ Loaded OMXHPI: 3523 rows
✓ Loaded OMXN40: 3631 rows
✓ Loaded OMXH25: 3526 rows
✓ Loaded STOXX50E: 3507 rows

Updated macro data shape: (3660, 14)
All columns: ['Date', 'EURUSD_return', 'EURSEK_return', 'EURCNY_return', 'GC_return', 'CL_return', 'VIX_return', 'GSPC_return', 'TNX_return', 'euribor_3m_diff', 'OMXHPI_return', 'OMXN40_return', 'OMXH25_return', 'STOXX50E_return']

Sample data with new variables:


,Date,EURUSD_return,EURSEK_return,EURCNY_return,GC_return,CL_return,VIX_return,GSPC_return,TNX_return,euribor_3m_diff,OMXHPI_return,OMXN40_return,OMXH25_return,STOXX50E_return
0,2012-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2012-01-03,-0.001466,0.001459,-0.003704,NaN,NaN,NaN,NaN,NaN,-0.010,0.014846,0.017512,0.015628,NaN
2,2012-01-04,0.008886,-0.002802,0.010452,0.007626,0.002525,-0.032651,0.000188,0.017857,-0.014,-0.013791,-0.010188,-0.013889,-0.016745
3,2012-01-05,-0.009574,-0.005720,-0.009308,0.004653,-0.013660,-0.033303,0.002944,-0.001003,-0.016,0.001091,0.004352,-0.002753,-0.014528
4,2012-01-06,-0.010440,0.000475,-0.010330,-0.002038,-0.002456,-0.039572,-0.002537,-0.016056,-0.015,NaN,0.001257,NaN,-0.007384
5,2012-01-09,-0.009065,-0.003231,-0.003144,-0.005321,-0.002462,0.021328,0.002262,-0.000510,-0.012,-0.007708,-0.001288,-0.006803,-0.005307
6,2012-01-10,0.007571,-0.002052,0.001970,0.014619,0.009180,-0.018035,0.008886,0.006122,-0.009,0.026436,0.017291,0.028233,0.026688
7,2012-01-11,-0.001879,0.000136,0.003272,0.005028,-0.013400,0.017400,0.000310,-0.034483,-0.010,-0.007343,-0.006305,-0.008995,-0.003391
8,2012-01-12,-0.002196,0.000715,-0.008011,0.004941,-0.017547,-0.027553,0.002337,0.015231,-0.012,0.007313,-0.000740,0.006414,0.002710
9,2012-01-13,0.007623,0.006866,0.005888,-0.010259,-0.004036,0.021495,-0.004948,-0.041386,-0.014,-0.008225,-0.006789,-0.009192,-0.003342


In [8]:
# Load bond yield data
# 1. Finland 10-year government bond yield
finland_10y_path = Path('./data/finland_10y.pkl')
if finland_10y_path.exists():
    df_finland_10y = pd.read_pickle(finland_10y_path)
    df_finland_10y['Date'] = pd.to_datetime(df_finland_10y['Date'])
    df_finland_10y = df_finland_10y.sort_values('Date')
    
    # Convert to numeric
    df_finland_10y['finland_10y_yield'] = pd.to_numeric(df_finland_10y['finland_10y_yield'], errors='coerce')
    
    # Calculate difference (not return)
    df_finland_10y['finland_10y_diff'] = df_finland_10y['finland_10y_yield'].diff()
    
    # Keep only Date and difference
    df_finland_10y = df_finland_10y[['Date', 'finland_10y_diff']]
    
    # Merge with macro_data
    macro_data = macro_data.merge(df_finland_10y, on='Date', how='left')
    print(f"✓ Loaded Finland 10y yield: {len(df_finland_10y)} rows")
else:
    print(f"⚠ File not found: {finland_10y_path}")

macro_data = macro_data.sort_values('Date').reset_index(drop=True)

print(f"\nUpdated macro data shape: {macro_data.shape}")
print(f"\nSample data with bond yields:")
macro_data.tail(10)

✓ Loaded Finland 10y yield: 3528 rows

Updated macro data shape: (3660, 15)

Sample data with bond yields:


/var/folders/vs/mq6j80s94v7_ts2x082jdx1w0000gn/T/ipykernel_2579/445923205.py:6: UserWarning: Parsing dates in %d.%m.%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df_finland_10y['Date'] = pd.to_datetime(df_finland_10y['Date'])


,Date,EURUSD_return,EURSEK_return,EURCNY_return,GC_return,CL_return,VIX_return,GSPC_return,TNX_return,euribor_3m_diff,OMXHPI_return,OMXN40_return,OMXH25_return,STOXX50E_return,finland_10y_diff
3650,2025-12-29,-0.001018,-0.004312,-0.001038,-0.045042,0.023616,0.044118,-0.003492,-0.004836,0.001,0.006691,0.000659,0.007421,0.000423,-0.03
3651,2025-12-30,0.000000,0.003959,-0.003143,0.010404,-0.002238,0.009155,-0.001376,0.003401,-0.003,0.006910,0.004020,0.006325,0.007739,0.01
3652,2025-12-31,-0.002208,0.000102,-0.003395,NaN,NaN,NaN,NaN,NaN,0.010,NaN,-0.000803,NaN,NaN,0.01
3653,2026-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.006249,NaN,0.005802,NaN,NaN
3654,2026-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.005313,NaN,0.005240,NaN,NaN
3655,2026-01-07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.009724,NaN,0.011199,NaN,NaN
3656,2026-01-08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.009064,NaN,-0.010344,NaN,NaN
3657,2026-01-09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.004594,NaN,0.005011,NaN,NaN
3658,2026-01-12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.001524,NaN,NaN
3659,2026-01-13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.000000,NaN,NaN


In [9]:
print(macro_data.isnull().sum())


Date                  0
EURUSD_return        17
EURSEK_return        15
EURCNY_return        15
GC_return           143
CL_return           148
VIX_return          142
GSPC_return         142
TNX_return          143
euribor_3m_diff     107
OMXHPI_return       138
OMXN40_return        30
OMXH25_return       135
STOXX50E_return     154
finland_10y_diff    133
dtype: int64


In [10]:
# Join macro variables to stock data by Date
# This will duplicate macro variables across all tickers for each date
stock_data = stock_data.merge(macro_data, on='Date', how='left')

print(f"Stock data shape after adding macro variables: {stock_data.shape}")
print(f"Columns: {stock_data.columns.tolist()}")
print(f"\nMissing values:")
print(stock_data.isnull().sum())
print(f"\nSample data with macro variables:")
stock_data.head(25)

Stock data shape after adding macro variables: (390127, 32)
Columns: ['ticker', 'Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'return', 'return_lag1', 'return_lag2', 'return_lag5', 'log_volume', 'gk_vol', 'momentum_12_1', 'amihud', 'open_to_close_return', 'close_to_open_return', 'EURUSD_return', 'EURSEK_return', 'EURCNY_return', 'GC_return', 'CL_return', 'VIX_return', 'GSPC_return', 'TNX_return', 'euribor_3m_diff', 'OMXHPI_return', 'OMXN40_return', 'OMXH25_return', 'STOXX50E_return', 'finland_10y_diff']

Missing values:
ticker                      0
Date                        0
Open                        0
High                        0
Low                         0
Close                       0
Adj Close                   0
Volume                      0
return                    162
return_lag1               324
return_lag2               486
return_lag5               972
log_volume                  0
gk_vol                      0
momentum_12_1           39887
amihud 

,ticker,Date,Open,High,Low,Close,Adj Close,Volume,return,return_lag1,...,CL_return,VIX_return,GSPC_return,TNX_return,euribor_3m_diff,OMXHPI_return,OMXN40_return,OMXH25_return,STOXX50E_return,finland_10y_diff
0,AALLON,2019-04-08,8.21,8.96,8.21,8.87,7.84,104367.0,NaN,NaN,...,0.020926,0.028081,0.001047,0.007197,0.000,-0.005252,0.000227,-0.007128,-0.002730,-0.02
1,AALLON,2019-04-09,8.90,8.90,8.46,8.60,7.60,23533.0,-0.030440,NaN,...,-0.006522,0.083460,-0.006067,-0.007940,0.000,-0.006051,-0.005364,-0.007585,-0.006062,0.00
2,AALLON,2019-04-10,8.60,8.60,8.47,8.47,7.48,11544.0,-0.015116,-0.030440,...,0.009847,-0.068627,0.003478,-0.008804,0.000,-0.002933,0.001375,-0.002990,0.002174,-0.01
3,AALLON,2019-04-11,8.45,8.50,8.35,8.50,7.51,6043.0,0.003542,-0.015116,...,-0.015942,-0.021053,0.000038,0.010900,0.000,0.001310,-0.000886,0.000388,0.003121,-0.03
4,AALLON,2019-04-12,8.50,8.55,8.35,8.50,7.51,3199.0,0.000000,0.003542,...,0.004876,-0.077573,0.006609,0.022364,0.000,0.002983,0.001793,0.003429,0.003636,0.04
5,AALLON,2019-04-15,8.60,8.60,8.50,8.50,7.51,1598.0,0.000000,0.000000,...,-0.007669,0.025812,-0.000629,-0.002734,0.000,-0.003893,-0.001132,-0.004919,0.000763,0.05
6,AALLON,2019-04-16,8.60,8.60,8.45,8.45,7.47,2226.0,-0.005882,0.000000,...,0.010252,-0.011364,0.000509,0.015276,0.000,-0.003293,-0.000019,-0.005289,0.003739,-0.01
7,AALLON,2019-04-17,8.45,8.55,8.45,8.55,7.56,2649.0,0.011834,-0.005882,...,-0.004528,0.034483,-0.002274,0.000000,-0.001,-0.004955,0.000893,-0.006772,0.004149,0.04
8,AALLON,2019-04-18,8.60,8.60,8.40,8.45,7.47,4214.0,-0.011696,0.011834,...,0.003764,-0.040476,0.001579,-0.012346,0.000,0.006896,0.003624,0.008142,0.006182,-0.04
9,AALLON,2019-04-23,8.45,8.50,8.35,8.50,7.51,3995.0,0.005917,-0.011696,...,0.009132,-0.011272,0.008841,-0.007722,0.000,0.003489,0.001123,0.003838,0.001320,0.02


## Add monthly macro (economic data) variables

In [11]:
# Load monthly economic data with lag structure
# These variables are released with a lag and change monthly

# 1. Finland Unemployment (1.5 month lag)
unemployment_path = Path('./data/finland_unemployment_daily.pkl')
if unemployment_path.exists():
    df_unemployment = pd.read_pickle(unemployment_path)
    df_unemployment['Date'] = pd.to_datetime(df_unemployment['Date'])
    
    # Merge with stock_data
    stock_data = stock_data.merge(df_unemployment, on='Date', how='left')
    print(f"✓ Loaded unemployment data: {len(df_unemployment)} rows")
    print(f"  Date range: {df_unemployment['Date'].min()} to {df_unemployment['Date'].max()}")
else:
    print(f"⚠ File not found: {unemployment_path}")

# 2. Finland CPI (1 month lag)
cpi_path = Path('./data/finland_cpi_daily.pkl')
if cpi_path.exists():
    df_cpi = pd.read_pickle(cpi_path)
    df_cpi['Date'] = pd.to_datetime(df_cpi['Date'])
    
    # Merge with stock_data
    stock_data = stock_data.merge(df_cpi, on='Date', how='left')
    print(f"✓ Loaded CPI data: {len(df_cpi)} rows")
    print(f"  Date range: {df_cpi['Date'].min()} to {df_cpi['Date'].max()}")
else:
    print(f"⚠ File not found: {cpi_path}")

# 3. Finland Consumer Confidence (1 month lag)
consumer_conf_path = Path('./data/finland_consumer_conf_daily.pkl')
if consumer_conf_path.exists():
    df_consumer_conf = pd.read_pickle(consumer_conf_path)
    df_consumer_conf['Date'] = pd.to_datetime(df_consumer_conf['Date'])
    
    # Merge with stock_data
    stock_data = stock_data.merge(df_consumer_conf, on='Date', how='left')
    print(f"✓ Loaded consumer confidence data: {len(df_consumer_conf)} rows")
    print(f"  Date range: {df_consumer_conf['Date'].min()} to {df_consumer_conf['Date'].max()}")
else:
    print(f"⚠ File not found: {consumer_conf_path}")

print(f"\nStock data shape after adding monthly economic variables: {stock_data.shape}")
print(f"\nNew columns added:")
print([col for col in stock_data.columns if 'unemp' in col or 'cpi' in col or 'consumer_conf' in col])
print(f"\nMissing values for new variables:")
print(stock_data[['unemp_rate_change_lagged', 'cpi_rate_change_lagged', 'consumer_conf_change_lagged']].isnull().sum())
print(f"\nSample data with monthly economic variables:")
stock_data.tail(10)

✓ Loaded unemployment data: 5418 rows
  Date range: 2011-02-01 00:00:00 to 2025-12-01 00:00:00
✓ Loaded CPI data: 5129 rows
  Date range: 2012-01-01 00:00:00 to 2026-01-15 00:00:00
✓ Loaded consumer confidence data: 5115 rows
  Date range: 2012-01-01 00:00:00 to 2026-01-01 00:00:00

Stock data shape after adding monthly economic variables: (390127, 35)

New columns added:
['unemp_rate_change_lagged', 'cpi_rate_change_lagged', 'consumer_conf_change_lagged']

Missing values for new variables:
unemp_rate_change_lagged       4409
cpi_rate_change_lagged            0
consumer_conf_change_lagged       0
dtype: int64

Sample data with monthly economic variables:


,ticker,Date,Open,High,Low,Close,Adj Close,Volume,return,return_lag1,...,TNX_return,euribor_3m_diff,OMXHPI_return,OMXN40_return,OMXH25_return,STOXX50E_return,finland_10y_diff,unemp_rate_change_lagged,cpi_rate_change_lagged,consumer_conf_change_lagged
390117,YIT,2025-12-12,3.208,3.258,3.206,3.226,3.226,88945.0,0.006866,0.007547,...,0.012799,-0.018,-0.009137,-0.004077,-0.009262,-0.005779,0.00,NaN,-0.17,0.9
390118,YIT,2025-12-15,3.226,3.230,3.016,3.016,3.016,388710.0,-0.065096,0.006866,...,-0.002861,-0.010,0.003017,-0.002636,0.003882,0.005560,-0.02,NaN,-0.05,0.9
390119,YIT,2025-12-16,3.020,3.078,3.020,3.042,3.042,241790.0,0.008621,-0.065096,...,-0.007891,-0.015,-0.000941,-0.007920,-0.001028,-0.006030,0.00,NaN,-0.05,0.9
390120,YIT,2025-12-17,3.042,3.074,3.010,3.048,3.048,207503.0,0.001972,0.008621,...,0.000482,-0.008,-0.002330,-0.003084,-0.003636,-0.006324,0.00,NaN,-0.05,0.9
390121,YIT,2025-12-18,3.048,3.128,3.042,3.128,3.128,116034.0,0.026247,0.001972,...,-0.008432,-0.014,0.012390,0.009020,0.013247,0.010567,0.00,NaN,-0.05,0.9
390122,YIT,2025-12-19,3.128,3.128,3.058,3.100,3.100,147037.0,-0.008951,0.026247,...,0.008503,-0.033,0.003268,0.008342,0.002758,0.003246,0.04,NaN,-0.05,0.9
390123,YIT,2025-12-22,3.082,3.114,3.064,3.100,3.100,178484.0,0.000000,-0.008951,...,0.004336,0.020,0.003356,-0.002979,0.003532,-0.002892,0.01,NaN,-0.05,0.9
390124,YIT,2025-12-23,3.094,3.132,3.090,3.104,3.104,139437.0,0.001290,0.000000,...,0.000000,-0.004,0.004018,0.017747,0.003188,0.000973,-0.04,NaN,-0.05,0.9
390125,YIT,2025-12-29,3.100,3.120,3.054,3.078,3.078,230871.0,-0.008376,0.001290,...,-0.004836,0.001,0.006691,0.000659,0.007421,0.000423,-0.03,NaN,-0.05,0.9
390126,YIT,2025-12-30,3.070,3.134,3.038,3.122,3.122,264079.0,0.014295,-0.008376,...,0.003401,-0.003,0.006910,0.004020,0.006325,0.007739,0.01,NaN,-0.05,0.9


## Tinkering

In [12]:
# Drop OHLCV columns
columns_to_drop = ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'OMXH25_return']
stock_data_clean = stock_data.drop(columns=columns_to_drop)

print(f"Dropped columns: {columns_to_drop}")
print(f"New shape: {stock_data_clean.shape}")
print(f"\nNull counts by column:")
print(stock_data_clean.isnull().sum())

Dropped columns: ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'OMXH25_return']
New shape: (390127, 28)

Null counts by column:
ticker                             0
Date                               0
return                           162
return_lag1                      324
return_lag2                      486
return_lag5                      972
log_volume                         0
gk_vol                             0
momentum_12_1                  39887
amihud                           324
open_to_close_return               0
close_to_open_return             162
EURUSD_return                    530
EURSEK_return                    348
EURCNY_return                    385
GC_return                      10684
CL_return                      11514
VIX_return                     10704
GSPC_return                    10704
TNX_return                     10798
euribor_3m_diff                 4214
OMXHPI_return                    237
OMXN40_return                    237
STOXX50E_re

In [13]:
stock_data_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 390127 entries, 0 to 390126
Data columns (total 28 columns):
 #   Column                       Non-Null Count   Dtype         
---  ------                       --------------   -----         
 0   ticker                       390127 non-null  object        
 1   Date                         390127 non-null  datetime64[ns]
 2   return                       389965 non-null  float64       
 3   return_lag1                  389803 non-null  float64       
 4   return_lag2                  389641 non-null  float64       
 5   return_lag5                  389155 non-null  float64       
 6   log_volume                   390127 non-null  float64       
 7   gk_vol                       390127 non-null  float64       
 8   momentum_12_1                350240 non-null  float64       
 9   amihud                       389803 non-null  float64       
 10  open_to_close_return         390127 non-null  float64       
 11  close_to_open_return      

In [14]:
float64cols = [c for c in stock_data_clean.columns if stock_data_clean[c].dtype == 'float64']
stock_data_clean[float64cols] = stock_data_clean[float64cols].astype('float32')

### Save to CSV and Parquet

In [15]:
# Save to CSV and Parquet
output_path = Path('data/stock_data_full.csv')
output_path.parent.mkdir(parents=True, exist_ok=True)
stock_data_clean.to_csv(output_path, index=False)
stock_data_clean.to_parquet(Path('data/stock_data_full.parquet'), compression='brotli')
print(f"\n✓ Saved stock_data to {output_path}")
print(f"  Shape: {stock_data_clean.shape}")
print(f"  Columns: {len(stock_data_clean.columns)}")


✓ Saved stock_data to data/stock_data_full.csv
  Shape: (390127, 28)
  Columns: 28
